# Fraud Detection

- [How a Fraud Decision System Works](#How-a-Fraud-Decision-System-Works)
  - [The threshold is a cost decision](#The-threshold-is-a-cost-decision)
  - [Labels arrive late](#Labels-arrive-late)
  - [The adversary adapts](#The-adversary-adapts)
  - [Hybrid scoring: a tree model plus LLM-derived features](#Hybrid-scoring:-a-tree-model-plus-LLM-derived-features)
  - [Shadow testing against a rule base](#Shadow-testing-against-a-rule-base)
- [Anomaly Detection](#Anomaly-Detection)
  - [Isolation Forest](#Isolation-Forest)
  - [One-Class SVM](#One-Class-SVM)
  - [Local Outlier Factor (LOF)](#Local-Outlier-Factor-(LOF))
  - [Comparing the three, when to use which](#Comparing-the-three,-when-to-use-which)
  - [Relevance to fraud detection specifically](#Relevance-to-fraud-detection-specifically)
  - [Evaluating an anomaly detector without labels](#Evaluating-an-anomaly-detector-without-labels)
- [Graph Features for Fraud Detection](#Graph-Features-for-Fraud-Detection)
- [Blogs](#Blogs)

### How a Fraud Decision System Works

A fraud system does not just output a probability. It turns a score into an action, and the pieces around the model matter as much as the model.

1. **Signals.** Device, IP, email, card, velocity (how many attempts per card, device or IP in the last hour), amount, merchant and history.
2. **Score.** A model returns a risk score inside the authorization path, so it has roughly 100 ms or less.
3. **Decision.** Thresholds and business rules (set per market, product or merchant) map the score to allow, step-up authentication (for example 3DS), manual review or block.
4. **Feedback.** Chargebacks, disputes and analyst decisions come back as labels, often weeks later.

#### The threshold is a cost decision

Fraud is rare (about 1 in 1,000 payments is often quoted), so accuracy is meaningless and the default 0.5 threshold is almost never right. Pick the threshold that minimizes expected cost: a missed fraud costs the transaction amount, a false alarm costs a review plus customer friction.

Synthetic scores where fraud scores high but overlaps with legitimate traffic:

In [1]:
import numpy as np, pandas as pd

rng = np.random.default_rng(0)
n, fraud_rate = 200_000, 0.002
y = rng.random(n) < fraud_rate
score = np.where(y, rng.beta(5, 2, n), rng.beta(1.2, 6, n))     # fraud scores high, legit low, with overlap
amount = rng.lognormal(4, 1, n)
cost_fp = 5.0                                                   # review + friction per false alarm

def stats(th):
    flag = score >= th
    cost = (amount * (y & ~flag)).sum() + cost_fp * (flag & ~y).sum()   # missed fraud loses the amount
    return cost, flag.mean(), (flag & y).sum() / max(flag.sum(), 1), (flag & y).sum() / y.sum()

ths = np.linspace(0.05, 0.95, 91)
costs = [stats(t)[0] for t in ths]
best = ths[int(np.argmin(costs))]
for name, th in [("default 0.5", 0.5), ("cost-optimal", best)]:
    cost, flagged, prec, rec = stats(th)
    print(f"{name:12s} threshold {th:.2f} | cost ${cost:>9,.0f} | flagged {flagged:5.1%} | precision {prec:5.1%} | recall {rec:5.1%}")

default 0.5  threshold 0.50 | cost $   26,181 | flagged  2.4% | precision  7.6% | recall 88.0%
cost-optimal threshold 0.62 | cost $   14,499 | flagged  0.6% | precision 25.4% | recall 72.5%


The cost-optimal threshold is not 0.5. Here it is higher: with only 0.2% fraud, the default threshold floods reviewers with false alarms (low precision), and each one costs money. If the average loss per missed fraud were larger, or reviews cheaper, the optimum would move lower. Change `cost_fp` or the amount distribution and the best threshold moves, which is why thresholds are tuned per merchant or market and revisited as costs change.

#### Labels arrive late

Chargebacks take weeks to arrive (30 to 120 days is common). Recent transactions therefore look cleaner than they are, because their fraud has not been reported yet. Evaluate and retrain on matured cohorts only, and expect recent weeks to look better than they will end up.

In [2]:
rng = np.random.default_rng(1)
weeks, per_week, true_rate = 20, 50_000, 0.002
n_fraud = int(weeks * per_week * true_rate)
tx_day = rng.integers(0, weeks * 7, n_fraud)                    # day each fraudulent transaction happened
report_day = tx_day + rng.integers(30, 121, n_fraud)            # day the chargeback is reported
today = weeks * 7
seen = report_day <= today
week = tx_day // 7
frac_seen = pd.Series(seen).groupby(week).mean()
age_weeks = (weeks - 1 - frac_seen.index)
print(pd.DataFrame({"weeks_old": age_weeks, "share_of_fraud_labelled_so_far": frac_seen.round(2)}).iloc[::3].to_string(index=False))

 weeks_old  share_of_fraud_labelled_so_far
        19                            1.00
        16                            0.96
        13                            0.75
        10                            0.53
         7                            0.27
         4                            0.07
         1                            0.00


A model evaluated on the last four weeks would see only a small share of the fraud that actually happened, so its measured recall and the apparent fraud rate are both biased. Some teams use earlier proxy labels (a customer report, an analyst decision) to shorten the delay, accepting that these are noisier.

#### The adversary adapts

Unlike most classification problems, the other side reacts to your defenses. Practical consequences: keep retraining on fresh data, watch score and feature drift, keep rules as a fast lever for new attack patterns, and test new models in shadow mode first so a change does not open a gap.

#### Hybrid scoring: a tree model plus LLM-derived features

A common pattern when fraud signals sit in free text (case notes, complaint narratives): an LLM reads the text and returns structured fields (a theme, a confidence), and a gradient-boosted tree model combines those with tabular features and makes the decision. The tree model stays the decision-maker because it is fast, auditable and handles a missing feature natively; the LLM is a feature extractor.

Design points: score in daily batch, not per request, so LLM latency and cost are manageable; cache outputs by input hash; pin the LLM and prompt version and monitor the distribution of its outputs, because a silent model update is a feature drift; make sure the text is available at prediction time (no leakage from information written after the outcome); and keep a fallback so that when the LLM is down the field is missing and the tree model still scores.

Synthetic cases where tabular features are weak and the LLM theme score carries extra signal:

In [1]:
import numpy as np
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

rng = np.random.default_rng(0)
n = 20000
y = rng.random(n) < 0.05
tabular = np.c_[rng.normal(0.4 * y, 1.0), rng.normal(0.2 * y, 1.0), rng.normal(size=n)]   # weak tabular signal
llm_theme = np.clip(0.25 + 0.45 * y + rng.normal(0, 0.2, n), 0, 1)                         # noisy theme probability from the LLM
idx_tr, idx_te = train_test_split(np.arange(n), test_size=0.3, random_state=0, stratify=y)

def auc_for(X, Xte=None):
    m = XGBClassifier(n_estimators=150, max_depth=3, learning_rate=0.08, verbosity=0).fit(X[idx_tr], y[idx_tr])
    return round(roc_auc_score(y[idx_te], m.predict_proba((X if Xte is None else Xte)[idx_te])[:, 1]), 3)

hybrid = np.c_[tabular, llm_theme]
print("tabular only        :", auc_for(tabular))
print("tabular + LLM theme :", auc_for(hybrid))
down = hybrid.copy(); down[idx_te[: len(idx_te) // 2], -1] = np.nan                        # LLM unavailable for half of the test cases
print("LLM down for half   :", auc_for(hybrid, down))

tabular only        : 0.603
tabular + LLM theme : 0.939
LLM down for half   : 0.632


#### Shadow testing against a rule base

Before replacing or supplementing rules, run the model in shadow mode: it scores the same live cases but takes no action. The sign-off evidence is not an AUC but a comparison on the cases that matter: what the model flags that the rules miss, what the rules flag that the model misses, and whether the extra flags are real. Because most extra flags have no label yet, an analyst reviews a sample of the model-only flags and you report the confirmed rate with its uncertainty, weighted by exposure.

Synthetic cases with an existing rule (large amounts on new accounts) and a model that also sees behavior:

In [2]:
import numpy as np
rng = np.random.default_rng(1)
n = 50000
exposure = rng.lognormal(7, 1.2, n)
new_account = rng.random(n) < 0.15
behavior = rng.normal(size=n)
p_fraud = 1 / (1 + np.exp(-(-6.5 + 0.9 * behavior + 1.2 * new_account + 0.35 * np.log(exposure / 1000))))
fraud = rng.random(n) < p_fraud

rule = (exposure > 5000) & new_account                          # the current rule base
score = 0.9 * behavior + 1.2 * new_account + 0.35 * np.log(exposure / 1000) + rng.normal(0, 0.8, n)
model = score >= np.quantile(score, 1 - rule.mean())            # model flags the same volume as the rules

both, only_rule, only_model = rule & model, rule & ~model, model & ~rule
print(f"flagged: rules {rule.sum()} | model {model.sum()} | both {both.sum()} | model-only {only_model.sum()} | rule-only {only_rule.sum()}")
sample = rng.choice(np.where(only_model)[0], 200, replace=False)               # analysts review a sample of model-only flags
print(f"confirmed fraud: model-only sample {fraud[sample].mean():.0%} | rule-only flags {fraud[only_rule].mean():.0%} | overall base rate {fraud.mean():.1%}")
print(f"exposure caught (fraud cases only): rules {exposure[rule & fraud].sum():,.0f} | model {exposure[model & fraud].sum():,.0f}")

flagged: rules 719 | model 719 | both 117 | model-only 602 | rule-only 602
confirmed fraud: model-only sample 2% | rule-only flags 0% | overall base rate 0.3%
exposure caught (fraud cases only): rules 87,634 | model 124,871


### Anomaly Detection

Anomaly detection asks "does this point fit the normal pattern?" instead of "which class is this?". It works with no labeled anomalies, or very few. This notebook covers Isolation Forest, One-Class SVM and Local Outlier Factor.

#### Isolation Forest

Anomalies are few and different, so random splits isolate them quickly. Build many random trees, each splitting on a random feature at a random threshold, and count how many splits it takes to isolate each point alone in its own partition. A shorter path means a more anomalous point.

Toy setup (1D): points [1, 2, 3, 4, 5, 50], where 50 is an obvious outlier.

**Worked isolation, by hand.**

Isolating 50: a first random split at, say, 25 (near the midpoint of the range) puts every normal point below it and 50 alone above it. Path length = 1.

Isolating 3, which sits in the dense cluster: the split at 25 does not help, since 1 to 5 all stay together. It takes splits inside the cluster:
```
split 1 (threshold 25): {1,2,3,4,5} stay together, 50 isolated (irrelevant to point 3 now)
split 2 (threshold 2.5, within the [1,5] range): separates {1,2} from {3,4,5}
split 3 (threshold 4.5): separates {3,4} from {5}
split 4 (threshold 3.5): separates {3} from {4}
```
Point 3 needs about 4 splits, while point 50 needed 1. Averaged over many random trees with different thresholds, the outlier consistently isolates fast and normal points take longer. That gap is the anomaly score.

In [3]:
import numpy as np
from sklearn.ensemble import IsolationForest

X = np.array([1, 2, 3, 4, 5, 50]).reshape(-1, 1)

iso = IsolationForest(n_estimators=100, random_state=42)
iso.fit(X)

scores = iso.decision_function(X)  # higher = more normal, lower/negative = more anomalous
preds = iso.predict(X)             # -1 = anomaly, 1 = normal

for x, s, p in zip(X.flatten(), scores, preds):
    print(f"x={x}: anomaly_score={s:.3f}, prediction={'anomaly' if p == -1 else 'normal'}")

x=1: anomaly_score=0.039, prediction=normal
x=2: anomaly_score=0.114, prediction=normal
x=3: anomaly_score=0.119, prediction=normal
x=4: anomaly_score=0.109, prediction=normal
x=5: anomaly_score=0.039, prediction=normal
x=50: anomaly_score=-0.258, prediction=anomaly


#### One-Class SVM

The same SVM machinery as classification (see `08-classical-ml.ipynb`), fit on only one class, the normal data. It learns a boundary that encloses the normal region as tightly as possible, and anything outside the boundary at prediction time is flagged. The parameter `nu` (analogous to the soft-margin C) is the expected fraction of outliers.

**Practical notes.**

An RBF kernel lets the boundary take any non-linear shape around the normal cluster, not just a hyperplane. Results are sensitive to `nu` and the kernel bandwidth (`gamma`): too tight a boundary flags normal variation, and too loose a boundary misses real anomalies. It works best when the normal class is one well-formed cluster. A multi-modal normal class (several distinct normal sub-patterns) is hard to enclose with a single boundary.

In [4]:
from sklearn.svm import OneClassSVM

ocsvm = OneClassSVM(nu=0.15, kernel="rbf", gamma="auto")
ocsvm.fit(X)

preds_svm = ocsvm.predict(X)
for x, p in zip(X.flatten(), preds_svm):
    print(f"x={x}: prediction={'anomaly' if p == -1 else 'normal'}")

x=1: prediction=anomaly
x=2: prediction=normal
x=3: prediction=anomaly
x=4: prediction=anomaly
x=5: prediction=normal
x=50: prediction=normal


#### Local Outlier Factor (LOF)

LOF is density-based, in the same spirit as DBSCAN (see `10-unsupervised.ipynb`), but it outputs a continuous anomaly score instead of hard core/border/noise labels. It compares a point's local density with the density of its neighbors. A point in a notably sparser region than its neighbors gets a high score, even if it is not globally far from everything.

**Worked by hand, k=2 neighbors.**

Toy points: [1, 2, 3, 10]. The key building block is reachability distance: reach-dist_k(p, o) = max(k-distance(o), dist(p, o)), where k-distance(o) is the distance from o to its own k-th nearest neighbor. The max is deliberate: it stops a point from getting an artificially small distance just because it lands very close to a neighbor that itself sits in a sparse area.

k-distances (k=2): k-distance(1)=2, k-distance(2)=1, k-distance(3)=2, k-distance(10)=8.

Local reachability density (lrd) is 1 divided by the average reachability distance to a point's k nearest neighbors:
```
lrd(1) = 1 / avg(reach-dist(1,2)=1, reach-dist(1,3)=2) = 1/1.5 = 0.667
lrd(2) = 1 / avg(reach-dist(2,1)=2, reach-dist(2,3)=2) = 1/2.0 = 0.500
lrd(3) = 1 / avg(reach-dist(3,2)=1, reach-dist(3,1)=2) = 1/1.5 = 0.667
lrd(10)= 1 / avg(reach-dist(10,3)=7, reach-dist(10,2)=8) = 1/7.5 = 0.133
```
LOF(p) is the average, over p's neighbors, of (neighbor's lrd / p's lrd):
```
LOF(3)  = avg(lrd(2)/lrd(3), lrd(1)/lrd(3)) = avg(0.75, 1.00) = 0.875   ~ 1, normal density
LOF(10) = avg(lrd(3)/lrd(10), lrd(2)/lrd(10)) = avg(5.01, 3.76) = 4.38  >> 1, clear outlier
```
LOF(3) = 0.875 means point 3's neighborhood is about as dense as its neighbors', so it is unremarkable. LOF(10) = 4.38 means its local density is roughly 4x lower than its neighbors', a clear outlier with a quantified strength instead of a binary flag.

**Why "local" matters.**

A single global density threshold (like DBSCAN's eps) fails when regions of the data have different natural densities: a point with normal density in a sparse region can look anomalous under a threshold tuned for a dense region elsewhere. LOF compares each point only with its own neighborhood, which is what "local" means.

**Practical notes.**

LOF suits data with clusters of varying density, which is DBSCAN's weak point. The cost is pairwise distance computation (as in KNN and DBSCAN), so it scales worse than Isolation Forest. The continuous score (`negative_outlier_factor_` in sklearn) lets you rank "most anomalous first".

In [5]:
from sklearn.neighbors import LocalOutlierFactor
import numpy as np

X_small = np.array([1, 2, 3, 10]).reshape(-1, 1)

lof_small = LocalOutlierFactor(n_neighbors=2)
lof_small.fit_predict(X_small)
lof_scores = -lof_small.negative_outlier_factor_  # sklearn negates it, flip back to match the hand-worked LOF values

for x, score in zip(X_small.flatten(), lof_scores):
    print(f"x={x}: LOF={score:.3f}")

x=1: LOF=0.875
x=2: LOF=1.333
x=3: LOF=0.875
x=10: LOF=4.375


In [6]:
from sklearn.neighbors import LocalOutlierFactor

lof = LocalOutlierFactor(n_neighbors=3)
preds_lof = lof.fit_predict(X)
scores_lof = lof.negative_outlier_factor_

for x, s, p in zip(X.flatten(), scores_lof, preds_lof):
    print(f"x={x}: LOF_score={s:.3f}, prediction={'anomaly' if p == -1 else 'normal'}")

x=1: LOF_score=-1.000, prediction=normal
x=2: LOF_score=-1.000, prediction=normal
x=3: LOF_score=-1.000, prediction=normal
x=4: LOF_score=-1.000, prediction=normal
x=5: LOF_score=-1.000, prediction=normal
x=50: LOF_score=-19.714, prediction=anomaly


#### Comparing the three, when to use which

| | Isolation Forest | One-Class SVM | LOF |
|---|---|---|---|
| Mechanism | random-split path length | learned boundary around normal region | local density ratio vs. neighbors |
| Handles varying density across the dataset | no, single global notion of "easy to isolate" | no, one boundary for the whole normal class | yes, this is exactly what it's built for |
| Scales to large data | best of the three, tree-based, no pairwise distances | moderate, kernel computation cost grows with n | worst, needs pairwise distances like KNN/DBSCAN |
| Output | anomaly score (path-length based) | hard in/out decision (or a score via decision_function) | continuous score, good for ranking |
| Best fit | large datasets, no strong density-variation concerns | normal class is unimodal/well-clustered, moderate size | normal data has multiple sub-clusters of different densities |

Rule of thumb: start with Isolation Forest as a quick, scalable baseline. Use LOF when the normal class is not one uniform density, which is common in fraud data (normal behavior for a high-volume merchant looks nothing like normal for a low-volume one). Use One-Class SVM when you have a clean, reasonably small, well-clustered normal class and want an explicit learned boundary.

#### Relevance to fraud detection specifically

These methods answer a different question than the classifiers in `08-classical-ml.ipynb` and `09-trees.ipynb`. A classifier asks "which typology is this?" and needs labeled examples of every typology. Anomaly detection asks "is this normal?" and needs no fraud labels. It is the right tool when fraud is rare or novel enough that you do not have, or do not trust, labeled examples. A new scam pattern would be misclassified by the fraud-theme-detection project's supervised model, which only knows its 10 typologies, but these methods could still flag it for not looking like normal activity.

Autoencoder reconstruction error, the deep-learning approach to anomaly detection, is in `03-autoencoders.ipynb`.

#### Evaluating an anomaly detector without labels

Have analysts review the top-k alerts (precision@k), inject synthetic anomalies, backtest against fraud confirmed later, and check stability across runs. Set the threshold by review capacity and the cost of a miss, not by a fixed contamination rate.

### Graph Features for Fraud Detection

Fraud rings often look normal per-transaction — what's abnormal is the *structure*: a tight mutual cluster, or unusual connectivity for the account type. Tabular, per-transaction features can't represent network position at all; graph topology features can.

**Topology Features: Degree, PageRank, Clustering Coefficient**

1. Nodes = accounts, edges = transactions — this directly models money flow, so centrality/community features map cleanly to "unusually connected in the flow of funds."
2. Degree alone doesn't cleanly separate a fraud ring from a legitimate high-degree hub (like a payroll processor) — a ring has degree ~9–11, a hub has degree 62, but degree alone isn't the discriminator.
3. Clustering coefficient is the real discriminator: a fraud ring is a dense mutual cluster (neighbors also connected to each other), while a star-shaped hub has clustering coefficient exactly 0, since its spokes never transact with each other.

In [7]:
import numpy as np
import networkx as nx
import pandas as pd

RNG = np.random.default_rng(42)
N = 1000

G = nx.barabasi_albert_graph(n=N, m=2, seed=42)   # realistic hub-and-spoke backbone

ring_nodes = list(range(500, 508))                # fraud ring: dense near-clique
for i in ring_nodes:
    for j in ring_nodes:
        if i != j:
            G.add_edge(i, j)

hub_node = 999                                    # legitimate hub: star, spokes don't interconnect
hub_spokes = RNG.choice([n for n in range(N) if n != hub_node and n not in ring_nodes], size=60, replace=False)
for s in hub_spokes:
    G.add_edge(hub_node, s)

degree = dict(G.degree())
pagerank = nx.pagerank(G, alpha=0.85)
clustering = nx.clustering(G)
topology_df = pd.DataFrame({
    "account_id": list(G.nodes()),
    "degree": [degree[n] for n in G.nodes()],
    "pagerank": [pagerank[n] for n in G.nodes()],
    "clustering_coeff": [clustering[n] for n in G.nodes()],
})

print(f"Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")
print("\nRing nodes (fraud):")
print(topology_df[topology_df.account_id.isin(ring_nodes)][["account_id", "degree", "clustering_coeff"]])
print("\nHub node (legitimate, high-degree):")
print(topology_df[topology_df.account_id == hub_node][["account_id", "degree", "clustering_coeff"]])

Nodes: 1000, Edges: 2084

Ring nodes (fraud):
     account_id  degree  clustering_coeff
500         500       9          0.583333
501         501      10          0.466667
502         502       9          0.583333
503         503       9          0.583333
504         504       9          0.583333
505         505      10          0.466667
506         506       9          0.583333
507         507      11          0.381818

Hub node (legitimate, high-degree):
     account_id  degree  clustering_coeff
999         999      62          0.003702


**Choosing Node/Edge Definitions — and Their Blind Spot**

1. Nodes = accounts, edges = transactions directly models money flow — but a ring that avoids direct transactions while sharing a device fingerprint or IP won't be connected by transaction edges at all.
2. Fix: add a second, shared-attribute edge type (device/IP) to catch that case — coordinated accounts hiding behind transaction-level separation become directly linkable once a second relationship type is added.

In [8]:
G_device = G.copy()
# Two accounts NOT directly transacting, but sharing a device ID (a device-fraud pattern transaction edges miss)
device_pair = (ring_nodes[0] + 150, ring_nodes[1] + 150)
dist_before = nx.shortest_path_length(G_device, *device_pair)

G_device.add_edge(*device_pair)   # the shared-device edge
dist_after = nx.shortest_path_length(G_device, *device_pair)

print(f"Accounts {device_pair}: shortest path via transactions only = {dist_before}")
print(f"Accounts {device_pair}: shortest path after adding a shared-device edge = {dist_after}")

Accounts (650, 651): shortest path via transactions only = 5
Accounts (650, 651): shortest path after adding a shared-device edge = 1


**Confirming the Ring Is Tabularly Invisible**

1. Train a tabular-only baseline alongside the combined (tabular + graph) model.
2. If tabular-only scores near the random baseline on ring detection, that's direct evidence the ring has no tabular signature by construction — adding graph features unlocks detection at all here, not an incremental improvement on an already-working baseline.

In [9]:
is_fraud = topology_df["account_id"].isin(ring_nodes).astype(int)
avg_txn_amount = RNG.lognormal(mean=7, sigma=1, size=N)
account_age_days = RNG.integers(30, 3000, size=N)
kyc_risk_score = RNG.uniform(0, 1, size=N)

full_df = topology_df.copy()
full_df["avg_txn_amount"] = avg_txn_amount
full_df["account_age_days"] = account_age_days
full_df["kyc_risk_score"] = kyc_risk_score
full_df["is_fraud"] = is_fraud

from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import average_precision_score

tabular_cols = ["avg_txn_amount", "account_age_days", "kyc_risk_score"]
graph_cols = ["degree", "pagerank", "clustering_coeff"]

X_train, X_test, y_train, y_test = train_test_split(
    full_df[tabular_cols + graph_cols], full_df["is_fraud"], test_size=0.3, stratify=full_df["is_fraud"], random_state=42
)

def fit_eval(cols, label):
    pos_rate = y_train.mean()
    m = XGBClassifier(n_estimators=200, max_depth=3, learning_rate=0.1,
                       scale_pos_weight=(1 - pos_rate) / pos_rate, eval_metric="logloss", random_state=42)
    m.fit(X_train[cols], y_train)
    proba = m.predict_proba(X_test[cols])[:, 1]
    print(f"{label}: PR-AUC = {average_precision_score(y_test, proba):.3f}")
    return m, proba

_, proba_tabular = fit_eval(tabular_cols, "Tabular-only baseline")
model_combined, proba_combined = fit_eval(tabular_cols + graph_cols, "Tabular + graph topology")

Tabular-only baseline: PR-AUC = 0.006


Tabular + graph topology: PR-AUC = 0.542


**Why the False-Positive Rate Stays Stable**

1. The usual precision/recall tradeoff happens when more recall comes from loosening the threshold on the *same* signal.
2. Graph features are orthogonal — a structurally different population, not a stronger version of the tabular signal — so the false-positive rate on the original tabular population doesn't move.
3. The PR-AUC gap from the previous cell (tabular-only near-chance, combined much higher) is the evidence: the combined model isn't pushing more borderline tabular cases over threshold, it's separating a population tabular features never scored highly at all.

**Validating the Signal Is Real, Not a Graph Artifact**

1. Direct check: does the legitimate hub score low despite high degree, because its clustering coefficient is 0?
2. Pair the score with SHAP attribution — "this account sits in a tightly connected cluster with a 0.58 clustering coefficient" is investigable; "score = 0.87" is not.

In [10]:
hub_row = full_df[full_df.account_id == hub_node]
hub_proba = model_combined.predict_proba(hub_row[tabular_cols + graph_cols])[0, 1]
print(f"Legitimate hub (degree={hub_row.degree.values[0]}, clustering={hub_row.clustering_coeff.values[0]:.3f}) "
      f"-> fraud probability: {hub_proba:.3f}")

ring_rows = full_df[full_df.account_id.isin(ring_nodes)]
ring_proba = model_combined.predict_proba(ring_rows[tabular_cols + graph_cols])[:, 1]
print(f"Fraud ring accounts -> mean fraud probability: {ring_proba.mean():.3f}")

import shap
explainer = shap.TreeExplainer(model_combined)
sv = explainer.shap_values(X_test)
mean_abs_shap = pd.Series(np.abs(sv).mean(axis=0), index=X_test.columns).sort_values(ascending=False)
print("\nMean |SHAP| by feature:")
print(mean_abs_shap)

Legitimate hub (degree=62, clustering=0.004) -> fraud probability: 0.006
Fraud ring accounts -> mean fraud probability: 0.874



Mean |SHAP| by feature:
clustering_coeff    4.781190
degree              2.852029
pagerank            0.392881
avg_txn_amount      0.048600
account_age_days    0.010536
kyc_risk_score      0.000000
dtype: float32


**Cold Start — A New Account With No History**

1. Degree and clustering are literally zero/undefined for a node with no edges — the model doesn't fail, but has no graph signal to work with, so scoring falls back entirely on whatever tabular features exist (which may also be thin for a new account).
2. A near-zero score for a zero-history account isn't necessarily safe — it means the model has no opinion, not that the account is verified clean. New accounts need a separate onboarding-risk control (KYC-based, not graph-based) until enough transaction history accumulates for graph features to mean anything.

In [11]:
G_cold = G.copy()
new_node = 1000
G_cold.add_node(new_node)   # zero transactions yet

new_degree = G_cold.degree(new_node)
new_clustering = nx.clustering(G_cold, new_node)
new_pagerank = nx.pagerank(G_cold, alpha=0.85)[new_node]
print(f"New account: degree={new_degree}, clustering={new_clustering}, pagerank={new_pagerank:.6f}")

cold_start_row = pd.DataFrame([{
    "avg_txn_amount": avg_txn_amount.mean(), "account_age_days": 0, "kyc_risk_score": 0.5,
    "degree": new_degree, "pagerank": new_pagerank, "clustering_coeff": new_clustering,
}])
cold_start_proba = model_combined.predict_proba(cold_start_row[tabular_cols + graph_cols])[0, 1]
print(f"Model's fraud probability for a brand-new, zero-history account: {cold_start_proba:.3f}")

New account: degree=0, clustering=0, pagerank=0.000150
Model's fraud probability for a brand-new, zero-history account: 0.000


### Blogs

- **Stripe Radar.** Stripe describes moving from simple models such as logistic regression to a Wide & Deep ensemble of XGBoost and a deep network, then in mid-2022 to a DNN-only architecture (a multi-branch design inspired by ResNeXt) that cut training time by over 85%, to under two hours. Radar scores each payment in under 100 ms using more than 1,000 characteristics, such as throwaway emails or the number of cards previously seen on an IP address. Stripe notes fraud is about 1 in 1,000 payments, and it exposes custom rules and Risk Insights (which features drove a score) on top of the model. [How we built it: Stripe Radar](https://stripe.dev/blog/how-we-built-it-stripe-radar.md)
- **Visa.** Visa describes AI and machine-learning models (Advanced Authorization, Deep Authorization and the Decision Manager risk score from 0 for low risk to 99 for high risk) built on attributes such as identity, velocity, geolocation and device intelligence. Visa stresses pairing the models with customizable business rules so that risk thresholds can differ by market, product or customer type, and says Decision Manager resolved 98.7% of the transactions it processed automatically, leaving humans the edge cases. [AI solutions for fraud prevention and detection](https://corporate.visa.com/en/solutions/visa-protect/insights/ai-fraud-detection.html)
- **Mastercard Decision Intelligence Pro.** Mastercard reports a recurrent neural network that assesses the relationships between the entities around a transaction and scores it in under 50 ms, with claimed detection improvements of 20% on average and up to 300% in some cases. These are Mastercard's own figures from initial modelling. [CNBC: Mastercard launches a GPT-like AI model to help banks detect fraud](https://www.cnbc.com/2024/02/01/mastercard-launches-gpt-like-ai-model-to-help-banks-detect-fraud.html)

Common pattern: a real-time score of roughly 50 to 100 ms, ML combined with tunable rules, network-scale data, models that moved from trees toward deep networks, and humans on the edge cases and strategy.